# Dataspace Catalog & Policy Configuration

This notebook configures the Provider's DCAT catalog, datasets, distributions, ODRL policies, and connector instances, followed by Consumer catalog discovery:
1. Executes prerequisite authentication and onboarding (`setup_auth.ipynb`).
2. Resolves the Provider's main catalog and data service.
3. Creates a DCAT dataset and two distributions (PULL via `http+asd` and PUSH via `http+pushing`).
4. Demonstrates ODRL policy creation via policy templates (`policy-templates/instantiate-odrl-offer`).
5. Demonstrates direct ODRL policy creation via the Management API (`/odrl-policies`).
6. Registers and instantiates a PULL connector template and instance (`NO_AUTH`).
7. Registers and instantiates a PUSH connector template and instance (`NO_AUTH`).
8. Executes Consumer catalog discovery against the Provider (DSP 2025-1 Catalog RPC).
9. Discovers the Consumer DSP callback endpoint from `/.well-known/dspace-version/2025-1`.

## 1. Prerequisites
Execute the authentication notebook to initialize participant DIDs, wallets, verifiable credentials, and mutual peer tokens.

In [1]:
%run ./setup_auth.ipynb

Authority URL: http://127.0.0.1:1500
Consumer URL:  http://127.0.0.1:1100
Provider URL:  http://127.0.0.1:1200
Wallet linked for Authority (http://127.0.0.1:1500)
Wallet linked for Consumer (http://127.0.0.1:1100)
Wallet linked for Provider (http://127.0.0.1:1200)
Authority DID: did:web:127.0.0.1%3A1500

Consumer DID:  did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ

Provider DID:  did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dE

## 2. Main Catalog & Data Service Resolution
Query the Provider's default catalog (`/catalogs/main`) and primary data service (`/data-services/main`) to obtain IDs for dataset and distribution associations.

In [2]:
url = data_space_provider + "/api/v1/catalog-agent/catalogs/main"
response = requests.request("GET", url, headers=json_header)
response_as_json = response.json()
catalog_id = response_as_json["id"]
print(json.dumps(response_as_json, indent=4))

{
    "dctConformsTo": null,
    "dctCreator": null,
    "dctIdentifier": "urn:catalog:441dee8a-8633-48fb-bb55-b20247193429",
    "dctIssued": "2026-10-08T06:29:46.175293Z",
    "dctModified": null,
    "dctTitle": null,
    "dspaceMainCatalog": true,
    "dspaceParticipantId": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "foafHomePage": null,
    "id": "urn:catalog:441dee8a-8633-48fb-bb55-b20247193429",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public"
}


In [3]:
url = data_space_provider + "/api/v1/catalog-agent/data-services/main"
response = requests.request("GET", url, headers=json_header)
response_as_json = response.json()
data_service_id = response_as_json["id"]
print(json.dumps(response_as_json, indent=4))

{
    "catalogId": "urn:catalog:441dee8a-8633-48fb-bb55-b20247193429",
    "dcatEndpointDescription": null,
    "dcatEndpointUrl": "http://127.0.0.1:1200/dsp/current",
    "dctConformsTo": null,
    "dctCreator": null,
    "dctDescription": null,
    "dctIdentifier": "urn:data-service:fb44e99b-de99-4ab4-bc70-26d630823e31",
    "dctIssued": "2026-10-08T06:29:46.190930Z",
    "dctModified": null,
    "dctTitle": null,
    "dspaceMainDataService": true,
    "id": "urn:data-service:fb44e99b-de99-4ab4-bc70-26d630823e31",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public"
}


## 3. Create Dataset & Distributions
Register a DCAT dataset under the catalog, then define two distributions:
- **PULL distribution**: format `http+asd` bound to the data service.
- **PUSH distribution**: format `http+pushing` bound to the data service.

In [4]:
url = data_space_provider + "/api/v1/catalog-agent/datasets"
payload = {
    "dctTitle": "My dataspace dataset",
    "catalogId": catalog_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
dataset_id = response_as_json["id"]
print(json.dumps(response_as_json, indent=4))

{
    "catalogId": "urn:catalog:441dee8a-8633-48fb-bb55-b20247193429",
    "dctConformsTo": null,
    "dctCreator": null,
    "dctDescription": null,
    "dctIdentifier": "urn:dataset:517c28ab-e284-47ef-a9ec-ee1275cca459",
    "dctIssued": "2026-10-12T22:06:08.210109Z",
    "dctModified": null,
    "dctTitle": "My dataspace dataset",
    "id": "urn:dataset:517c28ab-e284-47ef-a9ec-ee1275cca459",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private"
}


In [5]:
url = data_space_provider + "/api/v1/catalog-agent/distributions"
payload = {
    "dctTitle": "Distribution of my dataset",
    "dctDescription": "This is the distribution of my dataset",
    "dcatAccessService": data_service_id,
    "datasetId": dataset_id,
    "dctFormats": "http+asd"
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
distribution_pull_id = response_as_json["id"]
print(json.dumps(response_as_json, indent=4))

{
    "datasetId": "urn:dataset:517c28ab-e284-47ef-a9ec-ee1275cca459",
    "dcatAccessService": "urn:data-service:fb44e99b-de99-4ab4-bc70-26d630823e31",
    "dctDescription": "This is the distribution of my dataset",
    "dctFormat": "http+asd",
    "dctIssued": "2026-10-12T22:06:08.703285Z",
    "dctModified": null,
    "dctTitle": "Distribution of my dataset",
    "id": "urn:distribution:3e410eca-94c7-4fe9-bb29-ea276f1681fd",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private"
}


In [6]:
url = data_space_provider + "/api/v1/catalog-agent/distributions"
payload = {
    "dctTitle": "Distribution of a streaming dataset",
    "dctDescription": "This is the distribution of my streaming dataset",
    "dcatAccessService": data_service_id,
    "datasetId": dataset_id,
    "dctFormats": "http+pushing"
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
distribution_push_id = response_as_json["id"]
print(json.dumps(response_as_json, indent=4))

{
    "datasetId": "urn:dataset:517c28ab-e284-47ef-a9ec-ee1275cca459",
    "dcatAccessService": "urn:data-service:fb44e99b-de99-4ab4-bc70-26d630823e31",
    "dctDescription": "This is the distribution of my streaming dataset",
    "dctFormat": "http+pushing",
    "dctIssued": "2026-10-12T22:06:09.132530Z",
    "dctModified": null,
    "dctTitle": "Distribution of a streaming dataset",
    "id": "urn:distribution:ac35e1ab-223a-48cb-a594-cc63a72f3334",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private"
}


## 4. ODRL Policy Creation via Template
List available policy templates and instantiate an ODRL offer policy for the dataset by supplying template parameters.

In [7]:
url = data_space_provider + "/api/v1/catalog-agent/policy-templates"
response = requests.request("GET", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=4))

{
    "items": [
        {
            "author": "",
            "content": {
                "permission": [
                    {
                        "action": "use",
                        "constraint": [
                            {
                                "leftOperand": "bla",
                                "operator": "eq",
                                "rightOperand": "$bla"
                            }
                        ]
                    }
                ]
            },
            "date": "2026-10-08T06:29:46.199742Z",
            "description": "Mi example super policy",
            "id": "policy-1",
            "parameters": {
                "$bla": {
                    "dataType": "string",
                    "restrictions": {},
                    "ui": {
                        "hidden": false
                    }
                }
            },
            "title": "Mi example policy",
            "userId": "system",
            "userRo

In [8]:
url = data_space_provider + "/api/v1/catalog-agent/policy-templates/instantiate-odrl-offer"
payload = {
    "id": "policy-1",
    "version": "1.0",
    "parameters": {
        "$bla": "did:web:university.upm.es"
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research access for UPM: grants read access to did:web:university.upm.es valid until 2025-12-31."
}

response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=4))

{
    "createdAt": "2026-10-12T22:06:12.918422Z",
    "description": "Research access for UPM: grants read access to did:web:university.upm.es valid until 2025-12-31.",
    "entity": "urn:dataset:517c28ab-e284-47ef-a9ec-ee1275cca459",
    "entityType": "Dataset",
    "id": "urn:odrl-policy:c03f6aaf-7550-400d-a91b-6c30f7b6b923",
    "instantiationParameters": {
        "$bla": "did:web:university.upm.es"
    },
    "odrlOffer": {
        "permission": [
            {
                "action": "use",
                "constraint": [
                    {
                        "leftOperand": "bla",
                        "operator": "eq",
                        "rightOperand": "did:web:university.upm.es"
                    }
                ]
            }
        ]
    },
    "sourceTemplateId": "policy-1",
    "sourceTemplateVersion": "1.0",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private"
}


## 5. Direct ODRL Policy Creation
Create commercial and research ODRL offer policies directly via the Catalog Agent API without using templates, specifying permissions, actions, and temporal constraints.

In [9]:
url = data_space_provider + "/api/v1/catalog-agent/odrl-policies"
payload = {
    "odrlOffer": {
        "permission": [
            {
                "action": "use",
                "constraint": [
                    {
                        "leftOperand": "dateTime",
                        "operator": "gteq",
                        "rightOperand": "2025-01-01T00:00:00Z"
                    },
                    {
                        "leftOperand": "dateTime",
                        "operator": "lteq",
                        "rightOperand": "2025-12-31T23:59:59Z"
                    },
                    {
                        "leftOperand": "purpose",
                        "operator": "isA",
                        "rightOperand": "commercial"
                    }
                ]
            }
        ],
        "obligation": [
            {
                "action": "attribute",
                "constraint": [
                    {
                        "leftOperand": "recipient",
                        "operator": "isA",
                        "rightOperand": "public"
                    }
                ]
            }
        ],
        "prohibition": [
            {
                "action": "derive",
                "constraint": []
            }
        ]
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Commercial use licence valid for 2025: permits usage for commercial purposes with mandatory attribution. Derivation of the dataset is prohibited."
}

response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=4))

{
    "createdAt": "2026-10-12T22:06:13.752624Z",
    "description": "Commercial use licence valid for 2025: permits usage for commercial purposes with mandatory attribution. Derivation of the dataset is prohibited.",
    "entity": "urn:dataset:517c28ab-e284-47ef-a9ec-ee1275cca459",
    "entityType": "Dataset",
    "id": "urn:odrl-policy:7afdc189-e086-4899-bd28-8ccf30fd7004",
    "instantiationParameters": null,
    "odrlOffer": {
        "obligation": [
            {
                "action": "attribute",
                "constraint": [
                    {
                        "leftOperand": "recipient",
                        "operator": "isA",
                        "rightOperand": "public"
                    }
                ]
            }
        ],
        "permission": [
            {
                "action": "use",
                "constraint": [
                    {
                        "leftOperand": "dateTime",
                        "operator": "gteq",
     

In [10]:
url = data_space_provider + "/api/v1/catalog-agent/odrl-policies"
payload = {
    "odrlOffer": {
        "permission": [
            {
                "action": "read",
                "constraint": [
                    {
                        "leftOperand": "dateTime",
                        "operator": "gteq",
                        "rightOperand": "2025-03-01T00:00:00Z"
                    },
                    {
                        "leftOperand": "dateTime",
                        "operator": "lteq",
                        "rightOperand": "2025-09-01T00:00:00Z"
                    },
                    {
                        "leftOperand": "purpose",
                        "operator": "isA",
                        "rightOperand": "research"
                    },
                    {
                        "leftOperand": "count",
                        "operator": "lteq",
                        "rightOperand": "10000"
                    }
                ]
            }
        ],
        "obligation": [],
        "prohibition": [
            {
                "action": "distribute",
                "constraint": []
            },
            {
                "action": "reproduce",
                "constraint": []
            }
        ]
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research trial access (Mar–Sep 2025): read-only, limited to 10 000 requests, restricted to research purposes. Redistribution and reproduction are prohibited."
}

response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=4))

{
    "createdAt": "2026-10-12T22:06:14.267940Z",
    "description": "Research trial access (Mar\u2013Sep 2025): read-only, limited to 10 000 requests, restricted to research purposes. Redistribution and reproduction are prohibited.",
    "entity": "urn:dataset:517c28ab-e284-47ef-a9ec-ee1275cca459",
    "entityType": "Dataset",
    "id": "urn:odrl-policy:4b3924d3-7d50-45a4-b27f-513806b5090a",
    "instantiationParameters": null,
    "odrlOffer": {
        "obligation": [],
        "permission": [
            {
                "action": "read",
                "constraint": [
                    {
                        "leftOperand": "dateTime",
                        "operator": "gteq",
                        "rightOperand": "2025-03-01T00:00:00Z"
                    },
                    {
                        "leftOperand": "dateTime",
                        "operator": "lteq",
                        "rightOperand": "2025-09-01T00:00:00Z"
                    },
            

## 6. Configure PULL Connector Instance
Register a PULL connector template (`NO_AUTH`, HTTP GET data access) and instantiate it for the PULL distribution.

In [11]:
url = data_space_provider + "/api/v1/connector/templates"
payload = {
    "authentication": {
        "type": "NO_AUTH"
    },
    "interaction": {
        "mode": "PULL",
        "dataAccess": {
            "protocol": "HTTP",
            "urlTemplate": "{{__ACCESS_URL__}}",
            "method": "{{__ACCESS_METHODS__}}",
            "headers": "{{__ACCESS_HEADERS__}}"
        }
    },
    "parameters": [
        {
            "paramType": "STRING",
            "name": "ACCESS_URL",
            "title": "Url",
            "required": True
        },
        {
            "paramType": "VEC<STRING>",
            "name": "ACCESS_METHODS",
            "title": "Methods",
            "required": True
        },
        {
            "paramType": "MAP<STRING,STRING>",
            "name": "ACCESS_HEADERS",
            "title": "Headers",
            "required": True
        }
    ]
}

response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
connector_name = response_as_json["name"]
connector_version = response_as_json["version"]
print(json.dumps(response_as_json, indent=4))

{
    "name": "urn:connector-template:90cbcf80-1dff-40b5-b735-38573dfb2300",
    "author": "admin",
    "description": null,
    "version": "1.0",
    "createdAt": "2026-10-12T22:06:15.104872Z",
    "authentication": {
        "type": "NO_AUTH"
    },
    "interaction": {
        "mode": "PULL",
        "dataAccess": {
            "protocol": "HTTP",
            "urlTemplate": "{{__ACCESS_URL__}}",
            "method": "{{__ACCESS_METHODS__}}",
            "headers": "{{__ACCESS_HEADERS__}}",
            "bodyTemplate": null
        }
    },
    "parameters": [
        {
            "name": "ACCESS_URL",
            "title": "Url",
            "description": null,
            "paramType": "STRING",
            "required": true,
            "defaultValue": null
        },
        {
            "name": "ACCESS_METHODS",
            "title": "Methods",
            "description": null,
            "paramType": "VEC<STRING>",
            "required": true,
            "defaultValue": null
 

In [12]:
url = data_space_provider + "/api/v1/connector/instances"
payload = {
    "templateName": connector_name,
    "templateVersion": connector_version,
    "distributionId": distribution_pull_id,
    "metadata": {
        "description": "asasdasddas",
        "ownerId": "asdas"
    },
    "dryRun": False,
    "parameters": {
        "ACCESS_URL": "http://127.0.0.1:8081",
        "ACCESS_METHODS": ["GET", "HEAD"],
        "ACCESS_HEADERS": {
            "Content-type": "application/json"
        }
    }
}

response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=4))

{
    "id": "urn:connector-instance:c8de25ee-a722-449d-9674-991d21b2569d",
    "userId": "system",
    "name": "urn:connector-template:90cbcf80-1dff-40b5-b735-38573dfb2300",
    "author": null,
    "description": null,
    "version": "1.0",
    "createdAt": "2026-10-12T22:06:15.518301Z",
    "authenticationConfig": {
        "type": "NO_AUTH"
    },
    "interaction": {
        "mode": "PULL",
        "dataAccess": {
            "protocol": "HTTP",
            "urlTemplate": "http://127.0.0.1:8081",
            "method": [
                "GET",
                "HEAD"
            ],
            "headers": {
                "Content-type": "application/json"
            },
            "bodyTemplate": null
        }
    },
    "distributionId": "urn:distribution:3e410eca-94c7-4fe9-bb29-ea276f1681fd"
}


## 7. Configure PUSH Connector Instance
Register a PUSH connector template (`NO_AUTH`, HTTP subscribe/unsubscribe interaction) and instantiate it with target backend subscription URLs for the PUSH distribution.

In [13]:
url = data_space_provider + "/api/v1/connector/templates"
payload = {
    "authentication": {"type": "NO_AUTH"},
    "interaction": {
        "mode": "PUSH",
        "subscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__SUB_URL__}}",          # ← resuelto en instantiation
            "method": "POST",
            "headers": {},
            "bodyTemplate": json.dumps({
                "url": "{{__SYS_OWN_URL_DOCKER__}}{{__RUNTIME_INGRESS__}}", # ← resuelto en perform_subscribe
                "event_type": "hotel_waste_generated"
            })
        },
        "unsubscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__UNSUB_URL__}}/{{__RUNTIME_JSON_{.subscription.data.ID}__}}",         # ← resuelto en instantiation
            "method": "POST",
            "headers": {},
            "bodyTemplate": json.dumps({})
        }
    },
    "parameters": [
        {
            "paramType": "STRING", 
            "name": "SUB_URL",   
            "title": "Subscription URL",
            "required": True
        },
        {
            "paramType": "STRING", 
            "name": "UNSUB_URL", 
            "title": "Unsubscription URL",
            "required": True
        }
    ]
}


response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
connector_name = response_as_json["name"]
connector_version = response_as_json["version"]
print(json.dumps(response_as_json, indent=4))

{
    "name": "urn:connector-template:9c4df675-056e-4063-afde-7e3d1aeaeeec",
    "author": "admin",
    "description": null,
    "version": "1.0",
    "createdAt": "2026-10-12T22:06:16.183185Z",
    "authentication": {
        "type": "NO_AUTH"
    },
    "interaction": {
        "mode": "PUSH",
        "subscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__SUB_URL__}}",
            "method": "POST",
            "headers": {},
            "bodyTemplate": "{\"url\": \"{{__SYS_OWN_URL_DOCKER__}}{{__RUNTIME_INGRESS__}}\", \"event_type\": \"hotel_waste_generated\"}"
        },
        "unsubscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__UNSUB_URL__}}/{{__RUNTIME_JSON_{.subscription.data.ID}__}}",
            "method": "POST",
            "headers": {},
            "bodyTemplate": "{}"
        }
    },
    "parameters": [
        {
            "name": "SUB_URL",
            "title": "Subscription URL",
            "description": null,
    

In [14]:
url = data_space_provider + "/api/v1/connector/instances"
payload = {
    "templateName": connector_name,
    "templateVersion": connector_version,
    "distributionId": distribution_push_id,
    "parameters": {
        "SUB_URL":   "http://127.0.0.1:8082/subscriptions/subscribe",
        "UNSUB_URL": "http://127.0.0.1:8082/subscriptions/unsubscribe"
        # SYS_* NO se pasan aquí — se inyectan solos en runtime
    }
}


response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=4))

{
    "id": "urn:connector-instance:c4258e59-6974-4ac0-8a24-ae936a20fa46",
    "userId": "system",
    "name": "urn:connector-template:9c4df675-056e-4063-afde-7e3d1aeaeeec",
    "author": null,
    "description": null,
    "version": "1.0",
    "createdAt": "2026-10-12T22:06:16.483071Z",
    "authenticationConfig": {
        "type": "NO_AUTH"
    },
    "interaction": {
        "mode": "PUSH",
        "subscribe": {
            "protocol": "HTTP",
            "urlTemplate": "http://127.0.0.1:8082/subscriptions/subscribe",
            "method": "POST",
            "headers": {},
            "bodyTemplate": "{\"url\": \"http://host.docker.internal:1200{{__RUNTIME_INGRESS__}}\", \"event_type\": \"hotel_waste_generated\"}"
        },
        "unsubscribe": {
            "protocol": "HTTP",
            "urlTemplate": "http://127.0.0.1:8082/subscriptions/unsubscribe/{{__RUNTIME_JSON_{.subscription.data.ID}__}}",
            "method": "POST",
            "headers": {},
            "bodyTempla

## 8. Consumer Catalog Discovery (DSP 2025-1)
The Consumer queries the Provider's catalog using the DSP 2025-1 Catalog RPC endpoint (`/dsp/current/catalog/rpc/setup-catalog-request`).
Extract the matching dataset target ID, offer policy ID, and Provider DSP endpoint for subsequent contract negotiation.

In [15]:
url = data_space_consumer + "/dsp/current/catalog/rpc/setup-catalog-request"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "filter": [], 
    "noCache": True
}

response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
policy_id = response_as_json["response"]["dataset"][-1]["hasPolicy"][0]["@id"]
target_id = response_as_json["response"]["dataset"][-1]["@id"]
data_space_provider_dsp_endpoint = response_as_json["response"]["service"]["endpointURL"]
response_as_json


{'request': {'associatedAgentPeer': 'did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ',
  'filter': [],
  'noCache': True},
 'response': {'@context': ['https://w3id.org/dspace/2025/1/context.jsonld'],
  '@type': 'Catalog',
  '@id': 'urn:catalog:441dee8a-8633-48fb-bb55-b20247193429',
  'identifier': 'urn:catalog:441dee8a-8633-48fb-bb55-b20247193429',
  'issued': '2026-10-08T06:29:46.175293',
  'catalog': [],
  'dataset': [{'@type': 'Dataset',
    '@id': 'urn:dataset:aecfd1a5-7d2c-4bc0-b447-b6045d77f43c',
    'identifier': 'urn:dataset:ae

In [16]:
print("Selected dataset target: \n{}\n".format(target_id))
print("Selected policy id: \n{}\n".format(policy_id))
print("Provider negotiation URL for DSP 2025: \n{}\n".format(data_space_provider_dsp_endpoint))


Selected dataset target: 
urn:dataset:aecfd1a5-7d2c-4bc0-b447-b6045d77f43c

Selected policy id: 
urn:odrl-policy:c03f6aaf-7550-400d-a91b-6c30f7b6b923

Provider negotiation URL for DSP 2025: 
http://127.0.0.1:1200/dsp/current



## 9. Consumer DSP Callback Resolution
Query the Consumer's `/.well-known/dspace-version/2025-1` discovery endpoint to obtain its canonical callback URL for incoming DSP negotiation and transfer messages.

In [17]:
url = data_space_consumer + "/.well-known/dspace-version/2025-1"
response = requests.request("GET", url, headers=json_header)
response_as_json = response.json()
data_space_consumer_dsp_endpoint = data_space_consumer + response_as_json["path"]
print("Consumer callback address: \n{}\n".format(data_space_consumer_dsp_endpoint))


Consumer callback address: 
http://127.0.0.1:1100/dsp/current

